低波动持仓数量最终复算

资金20万元、万五佣金及最低5元；固定60日低波动信号、90%目标仓位、20日调仓。v3修正整数手浮点边界，v1及v2结果保留；未使用封存价格。

In [1]:
from pathlib import Path
import json,hashlib
import pandas as pd
import numpy as np
root=Path('/Users/guo/Documents/stocks/quant-research')
run=root/'artifacts/lowvol-position-count-20261013-v3'
verified=0
for version in [1,2,3]:
    folder=root/f'artifacts/lowvol-position-count-20261013-v{version}'
    receipt=json.loads((folder/'independent-audit.json').read_text())
    for path,digest in receipt['source_hashes'].items():
        if '/results/' in path or path.endswith(('protocol.json','implementation-freeze.json')):
            assert hashlib.sha256((root/path).read_bytes()).hexdigest()==digest,path
            verified+=1
    print({'version':version,'audited_accounts':receipt['accounts'],'max_NAV_error':max(x['max_nav_error'] for x in receipt['records']),'sealed_price_rows_read':receipt['sealed_price_rows_read']})
print('verified_bindings',verified)


{'version': 1, 'audited_accounts': 26, 'max_NAV_error': 1.1641532182693481e-10, 'sealed_price_rows_read': 0}
{'version': 2, 'audited_accounts': 18, 'max_NAV_error': 1.1641532182693481e-10, 'sealed_price_rows_read': 0}
{'version': 3, 'audited_accounts': 18, 'max_NAV_error': 5.820766091346741e-11, 'sealed_price_rows_read': 0}
verified_bindings 390


In [2]:
rows=pd.read_csv(run/'results/comparison.csv')
phase=rows[rows.scenario.isin(['base','phase5','phase10','phase15'])]
print(phase.pivot(index='position_count',columns='scenario',values='net_return').to_string())
print(phase.groupby('position_count').agg(min_return=('net_return','min'),median_return=('net_return','median'),worst_drawdown=('max_drawdown','max')).to_string())
for item in rows.itertuples():
    nav=pd.read_parquet(run/f'results/{item.scenario}/{item.strategy}/nav.parquet')
    v=np.r_[200000.,nav.equity]
    assert abs(v[-1]/200000-1-item.net_return)<1e-10
    assert abs(-(v/np.maximum.accumulate(v)-1).min()-item.max_drawdown)<1e-10
print('All 18 final net returns and daily drawdowns checked.')


scenario            base   phase10   phase15    phase5
position_count                                        
6               0.573219  0.129630  0.088341  0.310435
10              0.580015  0.139605  0.168474  0.288863
20              0.571700  0.059696  0.130040  0.288772
                min_return  median_return  worst_drawdown
position_count                                           
6                 0.088341       0.220033        0.226194
10                0.139605       0.228668        0.220757
20                0.059696       0.209406        0.219625
All 18 final net returns and daily drawdowns checked.


In [3]:
print(pd.DataFrame(json.loads((run/'stability-summary.json').read_text())).to_string(index=False))
parts=pd.DataFrame(json.loads((run/'attribution.json').read_text()))
print(parts[parts.scenario.eq('base')].to_string(index=False))
decision=json.loads((run/'decision.json').read_text())
print({'retain_for_open_research':decision['retain_for_open_research'],'sealed_test_run':decision['sealed_test_authorized_or_run'],'formal_profit_qualified':decision['formal_profit_qualified']})


 position_count  positive_phases  median_phase_return  min_phase_return  max_phase_return  phase_return_range  median_phase_cagr  median_phase_calmar  worst_phase_drawdown  median_phase_drawdown  double_cost_net_return  passes_prespecified_continuation_gate_relative_to_six
              6                4             0.220033          0.088341          0.573219            0.484878           0.053548             0.312099              0.226194               0.188703                0.433785                                                   True
             10                4             0.228668          0.139605          0.580015            0.440410           0.055838             0.356692              0.220757               0.167190                0.443063                                                   True
             20                4             0.209406          0.059696          0.571700            0.512004           0.051230             0.366392              0.219625       

10只为有条件继续研究对象，不是全新盲测的最优配置。起点相关、银行暴露、年度阶段及部分公司行动不明限制结论。完整口径、执行修正和费用在同名报告中。